# phase_2 — YOLO detector AUDIT (run after training)

Turns the YOLO concerns in `docs/critical_yolo.md` (= **B1**) into numbers, using the
trained `best.pt` + the built `yolo_ds` (NO M1/M3 needed). Tests:
1. per-class mAP · 2. **static-prior baseline** (YOLO must beat an image-blind mean-box
template) · 3. per-region IoU · 4. **stratified by anatomical atypicality** · 5. perturbation
(delete a region's pixels → does its box move?). The decisive **phep 3 (gold-vs-detector
oracle ablation at M3)** is DEFERRED — it needs M1 features + M3.

Run on a **GPU** session, *Run All*. Weights are pulled from Drive if not already local.

## CONFIG — edit these

In [ ]:
import os
PHASE2_SRC = "/kaggle/working/repo/phase_2"
IMAGES      = "/kaggle/input/datasets/nguynnghin/mimic-cxr-448/mimic-cxr-448"
YOLO_LABELS = "/kaggle/input/datasets/nguynnghin/yolo-labels"
RUN_NAME    = "det29"
SPLIT       = "val"      # val (silver) | test (silver minus gold). Gold needs a separate ds.
MAX_IMAGES  = 2000       # cap IoU eval for speed (0 = all)
DRIVE_FOLDER_ID = "1c6AJ3fjsL449kiMK4xiXfnzwzA4gIo0O"
RCLONE_REMOTE   = "dhint:phase2_runs"
for k,v in dict(PHASE2_SRC=PHASE2_SRC,IMAGES=IMAGES,YOLO_LABELS=YOLO_LABELS,RUN_NAME=RUN_NAME,
                SPLIT=SPLIT,MAX_IMAGES=MAX_IMAGES,DRIVE_FOLDER_ID=DRIVE_FOLDER_ID,
                RCLONE_REMOTE=RCLONE_REMOTE).items():
    os.environ[k]=str(v)
print("config set | RUN_NAME =",RUN_NAME,"| SPLIT =",SPLIT)

In [ ]:
# code from GitHub (Internet ON)
!rm -rf /kaggle/working/repo && git clone -q https://github.com/hiennguyendang/phase_2_3_4_5 /kaggle/working/repo
!rm -rf /kaggle/working/phase_2 && cp -r "$PHASE2_SRC" /kaggle/working/phase_2
%cd /kaggle/working/phase_2
!pip install -q ultralytics

In [ ]:
import torch
print("torch:",torch.__version__,"| cuda:",torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Settings -> Accelerator -> GPU T4."

## install rclone + auth (your OAuth token) + pull best.pt

In [ ]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working
  curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip
  cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1

In [ ]:
import os
# Drive via YOUR OAuth token (NOT a service account). Secret GDRIVE_TOKEN from `rclone authorize "drive"`.
SYNC_OK="0"
try:
    from kaggle_secrets import UserSecretsClient
    sec=UserSecretsClient()
    os.environ["RCLONE_CONFIG_DHINT_TYPE"]="drive"
    os.environ["RCLONE_CONFIG_DHINT_TOKEN"]=sec.get_secret("GDRIVE_TOKEN").strip()
    os.environ["RCLONE_CONFIG_DHINT_SCOPE"]="drive"
    os.environ["RCLONE_CONFIG_DHINT_ROOT_FOLDER_ID"]=os.environ["DRIVE_FOLDER_ID"]
    os.environ.pop("RCLONE_CONFIG_DHINT_SERVICE_ACCOUNT_FILE",None)
    for ks,ke in [("GDRIVE_CLIENT_ID","RCLONE_CONFIG_DHINT_CLIENT_ID"),("GDRIVE_CLIENT_SECRET","RCLONE_CONFIG_DHINT_CLIENT_SECRET")]:
        try: os.environ[ke]=sec.get_secret(ks).strip()
        except Exception: pass
    if os.system('rclone lsd dhint: >/dev/null 2>&1')==0:
        SYNC_OK="1"; print("remote OK ->",os.environ["RCLONE_REMOTE"])
except Exception as e:
    print("[WARN] no Drive:",e)
os.environ["SYNC_OK"]=SYNC_OK; print("SYNC_OK =",SYNC_OK)

In [ ]:
import os
# pull best.pt from Drive if not already local (same-session run already has it locally)
RUN=os.environ["RUN_NAME"]; W=f"/kaggle/working/runs/{RUN}/weights/best.pt"
if not os.path.exists(W) and os.environ.get("SYNC_OK")=="1":
    os.makedirs(os.path.dirname(W),exist_ok=True)
    os.system(f'rclone copy "{os.environ["RCLONE_REMOTE"]}/{RUN}/weights" /kaggle/working/runs/{RUN}/weights --include "best.pt"')
print("best.pt present:",os.path.exists(W),"->",W)
assert os.path.exists(W),"best.pt not found locally or on Drive — train first (or set RUN_NAME)."

## assemble yolo_ds (symlink images to the prebuilt labels; gold held out)

In [ ]:
import os, glob, zipfile
YL=os.environ["YOLO_LABELS"]
zips=glob.glob(os.path.join(YL,"**","labels.zip"),recursive=True)
LDIR=YL
if zips:
    LDIR="/kaggle/working/yolo_labels_ex"; os.makedirs(LDIR,exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z: z.extractall(LDIR)
os.environ["LDIR"]=LDIR
!python link_yolo_images.py --labels-dir "$LDIR" --images-root "$IMAGES" --out /kaggle/working/yolo_ds

## run the audit

In [ ]:
!python audit_yolo.py   --weights /kaggle/working/runs/$RUN_NAME/weights/best.pt   --yolo-ds /kaggle/working/yolo_ds --split $SPLIT --max-images $MAX_IMAGES   --out /kaggle/working/runs/$RUN_NAME/audit

In [ ]:
import os, json
RUN=os.environ["RUN_NAME"]
rep=f"/kaggle/working/runs/{RUN}/audit/audit_report.json"
print(open(rep).read())
if os.environ.get("SYNC_OK")=="1":
    os.system(f'rclone copy /kaggle/working/runs/{RUN}/audit "{os.environ["RCLONE_REMOTE"]}/{RUN}/audit"')
    print("
pushed audit -> %s/%s/audit"%(os.environ["RCLONE_REMOTE"],RUN))